# GSB 5544 — NumPy in 7 Questions  
**SOLUTION**

NumPy is the library for doing math on **many numbers at once**. We'll answer 7 questions about my coffee spending; each question introduces one NumPy idea.

**Data:** every coffee purchase from my bank statement (2018–2025), one row per purchase.

In [1]:
import numpy as np

data = np.genfromtxt("https://raw.githubusercontent.com/gato365/gsb5544_instructor_learn_prep/main/assignments/Data/coffee_numpy.csv", delimiter=",", names=True, dtype=None, encoding="utf-8")

amount = data["amount"]          # dollars (negative = money out)
year   = data["year"]
day    = data["day_of_week"]
amount[:5]

array([-20.1 ,  -7.43,  -2.75,  -2.75,  -9.21])

---
## Q1. How many coffees did I buy?  → the array

A NumPy **array** is a list of values that are **all the same type**. That one rule is what makes it fast.

In [2]:
print(type(amount))
print("how many :", amount.shape)      # (821,) -> 821 values, 1 dimension
print("what type:", amount.dtype)      # float64 -> decimal numbers

<class 'numpy.ndarray'>
how many : (821,)
what type: float64


In [3]:
# Arrays from scratch
print(np.array([3, 5, 2]))
print(np.arange(0, 10, 2))             # start, stop (exclusive), step
print(np.zeros(3))

[3 5 2]
[0 2 4 6 8]
[0. 0. 0.]


---
## Q2. Why is `day` text and `amount` a number?  → dtype

One array = one dtype. Mix types and NumPy picks the most general one — a single word turns a whole column into text.

In [4]:
print(np.array([1, 2, 3]).dtype)
print(np.array([1, 2.5, 3]).dtype)        # int promoted to float
print(np.array([1, 2, "three"]).dtype)    # everything becomes text!

int64
float64
<U21


In [5]:
# .astype() converts
print(year.dtype, "->", year.astype(str)[:3])

int64 -> ['2018' '2018' '2019']


---
## Q3. What would every coffee cost with 8% tax?  → vectorization

Math applies to **every element at once**. No loop.

In [6]:
spend = np.abs(amount)                 # make dollars positive
with_tax = spend * 1.08
with_tax[:5]

array([21.708 ,  8.0244,  2.97  ,  2.97  ,  9.9468])

In [7]:
# How much faster than a loop?
big = np.random.rand(1_000_000)
%timeit [x * 1.08 for x in big]
%timeit big * 1.08

45.83 ms per loop
0.17 ms per loop


---
## Q4. How much have I spent in total, and what was my biggest coffee?  → aggregation

In [8]:
print("total  :", spend.sum().round(2))
print("average:", spend.mean().round(2))
print("biggest:", spend.max())

total  : 4974.47
average: 6.06
biggest: 55.89


In [9]:
# Q: WHICH purchase was the biggest?  argmax = position of the max
i = spend.argmax()
print("row", i, "->", spend[i], "on a", day[i], "in", year[i])

row 703 -> 55.89 on a Monday in 2025


---
## Q5. How many weekend coffees cost more than $10?  → boolean masks

A comparison gives an array of `True`/`False` — a **mask**. `True` counts as 1, so `.sum()` counts matches, and `arr[mask]` keeps them.

In [10]:
over_10 = spend > 10
print(over_10[:5])
print("coffees over $10:", over_10.sum())

[ True False False False False]
coffees over $10: 83


In [11]:
weekend = (day == "Saturday") | (day == "Sunday")     # | = or, & = and, ~ = not
print("weekend coffees:", weekend.sum())
print("weekend coffees over $10:", (weekend & over_10).sum())

weekend coffees: 26
weekend coffees over $10: 7


In [12]:
# Q: What did those weekend splurges cost?
spend[weekend & over_10]

array([11.  , 11.  , 10.7 , 12.  , 17.5 , 32.24, 13.05])

In [13]:
# Q: What fraction of my coffees are on Mondays?
(day == "Monday").mean()

np.float64(0.4543239951278928)

---
## Q6. Was each coffee "small" or "large"?  → `np.where`

`np.where(condition, value_if_true, value_if_false)` is an if/else applied to every element — it turns a number into a category.

In [14]:
size = np.where(spend > 10, "large", "small")
print(size[:8])
print("large:", (size == "large").sum(), " small:", (size == "small").sum())

['large' 'small' 'small' 'small' 'small' 'small' 'large' 'small']
large: 83  small: 738


In [15]:
# Q: Which purchases (positions) were over $15?
np.where(spend > 15)[0]

array([  0,  65,  79, 176, 179, 188, 413, 428, 564, 608, 623, 625, 683,
       697, 703, 725, 727, 749, 818])

---
## Q7. Why did my average come out as `nan`?  → missing values

`np.nan` means "missing". Regular math **propagates** it; the `nan*` functions **skip** it.

In [16]:
x = np.array([4.5, np.nan, 3.0, 7.25])
print(np.mean(x))          # nan  -> one missing value poisons the result
print(np.nanmean(x))       # skips the missing value

nan
4.916666666666667


In [17]:
# Q: How do I find the missing values?   (nan != nan, so use isnan)
print(np.isnan(x))
print(np.isnan(x).sum(), "missing")
print("any missing in spend?", np.isnan(spend).any())

[False  True False False]
1 missing
any missing in spend? False


✅ **Check:** how many coffees over $5 did I buy on Fridays in 2024? (one line, using masks)

In [18]:
# your answer here
((spend > 5) & (day == "Friday") & (year == 2024)).sum()

np.int64(5)

---
## Summary

| Question | NumPy idea |
|---|---|
| How many coffees? | array, `.shape`, `.dtype` |
| Why is one column text? | one dtype per array, `.astype()` |
| Cost with tax? | vectorized math |
| Total / biggest? | `.sum()`, `.mean()`, `.max()`, `.argmax()` |
| How many weekend coffees over $10? | boolean masks, `&` `\|` `~`, `.sum()` |
| Small or large? | `np.where` |
| Why `nan`? | `np.isnan`, `np.nanmean` |